<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_YB_WEEK_10_2_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 수상작 리뷰 1

서울시 따릉이 대여량 예측 경진대회

https://dacon.io/competitions/open/235576/codeshare/1545?page=1&dtype=recent


## 주제
따릉이 모델 평가프로그램





## 데이터

train.csv / test.csv (따릉이 대여량 및 기상 데이터)
- id : 고유 id
- hour : 시간
- temperature : 기온
- precipitation : 비가 오지 않았으면 0, 비가 오면 1
windspeed 풍속(평균)
- humidity : 습도
- visibility : 시정(視程), 시계(視界)(특정 기상 상태에 따른 가시성을 의미)
- ozone : 오존
- pm10 : 미세먼지(머리카락 굵기의 1/5에서 1/7 크기의 미세먼지)
- pm2.5 : 미세먼지(머리카락 굵기의 1/20에서 1/30 크기의 미세먼지)
- count : 시간에 따른 따릉이 대여 수

## 코드 흐름


### 1. 데이터 탐색 (EDA)

- info(), describe() 등을 활용해 데이터의 결측치 및 기술 통계량, 이상치(IQR 기준)를 파악함.

- matplotlib과 seaborn 라이브러리를 활용해 lmplot, boxplot, violinplot, pairplot 등 다양한 시각화를 진행. 시간대별 대여량 추이 및 습도, 가시거리 등 기상 변수 간의 상관관계를 다각도로 분석함.

### 2. 데이터 전처리 (Data Cleansing)

- isna().sum()으로 결측치를 확인한 후, fillna() 함수를 활용하여 결측값을 각 변수의 평균값(mean())으로 대체함.

### 3. 변수 선택 및 초기 모델 구축 (Feature Engineering & Initial Modeling)

- 분석에 유의미하다고 판단된 5개의 컬럼만 추출하여 학습 데이터를 세팅함.

- 예측을 위한 베이스 모델로 KNN (K-Nearest Neighbors)과 Random Forest 회귀 모델을 구축함.

### 4. 모델 학습 및 검증 (Model Tuning & Evaluation)

- 5개 모델 성능 비교: Decision Tree, Random Forest, LightGBM, XGBoost, KNN을 딕셔너리로 묶어 반복문(for문)으로 한 번에 학습시킴.

- 교차 검증 (Cross Validation): 홀드아웃 기법(train_test_split)과 K-Fold 교차 검증(5-Fold, 10-Fold)을 적용하여 모델의 과적합을 방지함. cross_val_score를 통해 구한 Negative MSE 값을 기준으로 모델별 성능을 바 차트(Bar plot)로 시각화하여 가장 우수한 모델을 판별함.

**주요 코드**

In [ ]:
# 5개의 회귀 모델을 딕셔너리 형태로 구축
model_dict = {'DT':DecisionTreeRegressor(), 'RF':RandomForestRegressor(),
              'LGB':lgb.LGBMRegressor(), 'XGB':xgb.XGBRegressor(), 'KNN':KNeighborsRegressor()}

# 5-Fold 교차 검증 설정
k_fold = KFold(n_splits=5, shuffle= True, random_state=10)

# for문을 활용하여 구축한 여러 모델들의 성능을 한 번에 평가
score = {}
for model_name in model_dict.keys():
    model = model_dict[model_name]
    score[model_name] = np.mean(cross_val_score(model, X_train, y_train,
                                                scoring = 'neg_mean_squared_error', n_jobs = -1, cv = k_fold))

## 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- 사용할 여러 머신러닝 모델을 딕셔너리(model_dict)에 담아두고, for문을 돌려 K-Fold 교차 검증 결과를 한 번에 추출하는 코드가 매우 인상적이었다. 실무나 대회에서 모델 성능을 빠르게 테스트하고 비교할 때 유용하게 쓰일 팁을 배웠다.

- 단순히 데이터를 학습/테스트 셋으로 한 번 나누는 홀드아웃(Hold-out) 방식뿐만 아니라, K-Fold 분할을 통해 모든 데이터가 최소 한 번씩 검증에 쓰이도록 하여 모델의 신뢰도와 일반화 성능을 객관적으로 평가하는 과정을 깊이 이해할 수 있었다.